In [77]:
import pandas as pd
import numpy as np

In [ ]:
# Load cleaned data 
df = pd.read_csv("cleaned_data.csv")

In [79]:
df.head()

,location_id,date,weather_code (wmo code),temperature_2m_max (°C),temperature_2m_min (°C),temperature,apparent_temperature_max (°C),apparent_temperature_min (°C),apparent_temperature_mean (°C),daylight_duration (s),sunshine_duration (s),precipitation,rainfall,precipitation_hours (h),wind_speed
0,0,1/1/2010,1,30.1,22.6,26.0,34.5,25.0,29.0,42220.20,38905.73,0.0,0.0,0,12.2
1,0,1/2/2010,51,30.1,23.7,26.3,33.9,26.1,29.7,42225.71,37451.01,0.1,0.1,1,13.0
2,0,1/3/2010,51,29.6,23.1,26.0,34.5,26.2,29.9,42231.68,33176.43,0.6,0.6,3,12.3
3,0,1/4/2010,2,28.9,23.1,25.7,31.7,26.1,28.4,42238.11,38289.20,0.0,0.0,0,17.0
4,0,1/5/2010,1,28.1,21.3,24.6,30.0,22.9,26.2,42244.99,39113.82,0.0,0.0,0,18.7


In [ ]:
# Convert 'date' to datetime and create time features
df['date'] = pd.to_datetime(df['date'])
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek  # Monday=0, Sunday=6
df['day_of_year'] = df['date'].dt.dayofyear
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

In [81]:
# Season encoding (Northern Hemisphere)
def get_season(month):
    if month in [12, 1, 2]:
        return 0  # Winter
    elif month in [3, 4, 5]:
        return 1  # Spring
    elif month in [6, 7, 8]:
        return 2  # Summer
    else:
        return 3  # Autumn

df['season'] = df['month'].apply(get_season)

In [ ]:
# Cyclical encoding for month, day_of_week, day_of_year
df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
df['month_cos'] = np.cos(2 * np.pi * df['month']/12)

df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week']/7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week']/7)

df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year']/365)
df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year']/365)

In [ ]:
# Sort by location and date before creating lag features
df.sort_values(by=['location_id', 'date'], inplace=True)

In [ ]:
# Create lag features
lag_features = ['temperature', 'rainfall', 'wind_speed', 'precipitation']
lag_days = [1, 2, 3, 7]

for feature in lag_features:
    for lag in lag_days:
        df[f'{feature}_lag{lag}'] = df.groupby('location_id')[feature].shift(lag)

In [ ]:
# Convert categorical features
df['location_id'] = df['location_id'].astype('category')
df['day_of_week'] = df['day_of_week'].astype('category')

In [ ]:
# Drop rows with NaN values caused by lagging 
df.dropna(inplace=True)

In [87]:
TARGETS  = ['temperature', 'rainfall', 'wind_speed', 'precipitation']
FEATURES = [col for col in df.columns if col not in TARGETS + ['date']]

In [ ]:
# Save the final feature-engineered dataset
df.to_csv("feature_engineered_data.csv", index=False)

In [89]:
df.head()

,location_id,date,weather_code (wmo code),temperature_2m_max (°C),temperature_2m_min (°C),temperature,apparent_temperature_max (°C),apparent_temperature_min (°C),apparent_temperature_mean (°C),daylight_duration (s),...,rainfall_lag3,rainfall_lag7,wind_speed_lag1,wind_speed_lag2,wind_speed_lag3,wind_speed_lag7,precipitation_lag1,precipitation_lag2,precipitation_lag3,precipitation_lag7
7,0,2010-01-08,53,28.4,23.4,25.6,32.2,26.1,29.0,42268.61,...,0.0,0.0,13.5,17.1,18.7,12.2,0.0,0.0,0.0,0.0
8,0,2010-01-09,61,29.1,23.5,26.0,35.4,28.3,31.3,42277.55,...,0.0,0.1,13.0,13.5,17.1,13.0,0.8,0.0,0.0,0.1
9,0,2010-01-10,51,30.1,22.6,26.2,35.8,27.1,31.2,42287.01,...,0.0,0.6,11.5,13.0,13.5,12.3,3.2,0.8,0.0,0.6
10,0,2010-01-11,51,29.7,22.4,26.3,35.5,26.5,31.3,42296.96,...,0.8,0.0,6.9,11.5,13.0,17.0,0.1,3.2,0.8,0.0
11,0,2010-01-12,63,29.8,24.2,26.3,36.7,28.9,31.7,42307.39,...,3.2,0.0,6.4,6.9,11.5,18.7,0.7,0.1,3.2,0.0


In [90]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 142182 entries, 7 to 142370
Data columns (total 43 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   location_id                     142182 non-null  category      
 1   date                            142182 non-null  datetime64[ns]
 2   weather_code (wmo code)         142182 non-null  int64         
 3   temperature_2m_max (°C)         142182 non-null  float64       
 4   temperature_2m_min (°C)         142182 non-null  float64       
 5   temperature                     142182 non-null  float64       
 6   apparent_temperature_max (°C)   142182 non-null  float64       
 7   apparent_temperature_min (°C)   142182 non-null  float64       
 8   apparent_temperature_mean (°C)  142182 non-null  float64       
 9   daylight_duration (s)           142182 non-null  float64       
 10  sunshine_duration (s)           142182 non-null  float64     

In [91]:
df.head()

,location_id,date,weather_code (wmo code),temperature_2m_max (°C),temperature_2m_min (°C),temperature,apparent_temperature_max (°C),apparent_temperature_min (°C),apparent_temperature_mean (°C),daylight_duration (s),...,rainfall_lag3,rainfall_lag7,wind_speed_lag1,wind_speed_lag2,wind_speed_lag3,wind_speed_lag7,precipitation_lag1,precipitation_lag2,precipitation_lag3,precipitation_lag7
7,0,2010-01-08,53,28.4,23.4,25.6,32.2,26.1,29.0,42268.61,...,0.0,0.0,13.5,17.1,18.7,12.2,0.0,0.0,0.0,0.0
8,0,2010-01-09,61,29.1,23.5,26.0,35.4,28.3,31.3,42277.55,...,0.0,0.1,13.0,13.5,17.1,13.0,0.8,0.0,0.0,0.1
9,0,2010-01-10,51,30.1,22.6,26.2,35.8,27.1,31.2,42287.01,...,0.0,0.6,11.5,13.0,13.5,12.3,3.2,0.8,0.0,0.6
10,0,2010-01-11,51,29.7,22.4,26.3,35.5,26.5,31.3,42296.96,...,0.8,0.0,6.9,11.5,13.0,17.0,0.1,3.2,0.8,0.0
11,0,2010-01-12,63,29.8,24.2,26.3,36.7,28.9,31.7,42307.39,...,3.2,0.0,6.4,6.9,11.5,18.7,0.7,0.1,3.2,0.0


In [ ]:
# List of categorical features for CatBoost
cat_features = ['location_id', 'day_of_week']

print("Feature Engineering complete.")
print(f"Final shape: {df.shape}")
print(f"CatBoost categorical features: {cat_features}")

Feature Engineering complete.
Final shape: (142182, 43)
CatBoost categorical features: ['location_id', 'day_of_week']
